In [1]:
!pip install -q transformers accelerate pypdf
!pip install -U bitsandbytes

from transformers import AutoTokenizer, AutoModelForCausalLM
from pypdf import PdfReader
import torch, re, json
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.2 MB/s eta 0:00:00


In [ ]:
from google.colab import files
from pathlib import Path
import re, json

# Option A: upload PDF from your machine
print("Upload your PDF…")
up = files.upload()  # pick your file in the dialog
pdf_name = next(iter(up.keys()))
PDF_PATH = Path(pdf_name)

Upload your PDF…


Saving rag_base.pdf to rag_base.pdf


In [ ]:
from pypdf import PdfReader

def pdf_to_text(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n\n".join((p.extract_text() or "") for p in reader.pages)

def clean_text(t: str) -> str:
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)                 # join hyphen line-breaks
    t = re.sub(r"[ \t]*\n(?!\s*\n)", " ", t)               # collapse single newlines
    t = re.sub(r"\s+\n", "\n", t)
    return t.strip()

raw = pdf_to_text(PDF_PATH)
text = clean_text(raw)

# Simple "chapter" split: split on common headings; fallback to big chunks if no headings
parts = re.split(r"(?i)(?:\n\s*(?:Kapitel|Chapter)\s+\d+\b|^\s*\d+(?:\.\d+)*\s+[^\n]+$)", text, flags=re.M)
parts = [p.strip() for p in parts if len(p.split()) > 80]  # keep only substantive parts
print(f"Segments detected: {len(parts)}")


Segments detected: 68


In [ ]:
# --- Load Qwen chat model in 4-bit and a helper to do chat-style generation ---
import torch, json, re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16
)

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb)

def chat_generate(system_prompt: str, user_prompt: str,
                  max_new_tokens=640, temperature=0.4, do_sample=True):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.05,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0], skip_special_tokens=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# Minimal Q&A generation without JSON parsing (Colab-ready)

!pip install -q transformers accelerate bitsandbytes pypdf

import re, json, torch
from pathlib import Path
from pypdf import PdfReader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

PDF_PATH = Path("rag_base.pdf")  # <-- set your PDF filename here

# --- PDF -> text -> segments ---
def pdf_to_text(path: Path) -> str:
    r = PdfReader(str(path))
    return "\n\n".join((p.extract_text() or "") for p in r.pages)

def clean_text(t: str) -> str:
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)          # join hyphen line-breaks
    t = re.sub(r"[ \t]*\n(?!\s*\n)", " ", t)        # collapse single newlines
    return t.strip()

text = clean_text(pdf_to_text(PDF_PATH))

# simple heading/size-based split + light content filter
parts = re.split(r"(?mi)(?:\n\s*(?:Kapitel|Chapter)\s+\d+\b|^\s*\d+(?:\.\d+)*\s+[^\n]+$)", text)
parts = [p.strip() for p in parts if len(p.split()) > 60]

BAD = ("Liebe Leserin, lieber Leser","Bundesminister","kostenlos herausgegeben",
       "Wahlwerbern","Europa-, Bundestags-","Wahl","Vorwort","Grußwort")
GOOD = ("Versicherung","Krankenversicherung","Beitrag","Pflicht","Bemessungsgrenze",
        "Mitglied","GKV","Krankenkasse","Leistung","Richtlinie","PKV")
def is_content(seg: str) -> bool:
    s = seg.lower()
    if any(b.lower() in s for b in BAD) and sum(g.lower() in s for g in GOOD) < 2:
        return False
    return len(seg.split()) >= 80 and any(g.lower() in s for g in GOOD)

segments = [p for p in parts if is_content(p)]
if not segments:
    segments = [text[i:i+2600] for i in range(0, len(text), 2600) if len(text[i:i+2600].split())>80]
print(f"Using {len(segments)} segments.")

# --- Load small chat model (4-bit) ---
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb)

def chat(system_prompt: str, user_prompt: str, max_new_tokens=650, temperature=0.4):
    msgs = [{"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt").to(model.device)
    y = model.generate(**x, max_new_tokens=max_new_tokens, temperature=temperature,
                       do_sample=False, top_p=0.9,
                       eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id)
    return tok.decode(y[0], skip_special_tokens=True)

# --- Prompt: force Q/A lines only (few-shot in same format) ---
FEWSHOT = (
    "Q: Was bedeutet Versicherungspflicht?\n"
    "A: Sie ist die gesetzliche Verpflichtung, krankenversichert zu sein.\n"
    "Q: Wer ist pflichtversichert?\n"
    "A: Arbeitnehmer, Studierende und Rentner, sofern gesetzliche Kriterien erfüllt sind.\n"
)

def make_prompts(section: str, n=10):
    section = section[:3000]
    system = ("Du erstellst aus einem deutschen Sachtext sinnvolle Prüfungsfragen mit Antworten. "
              "Nutze ausschließlich Informationen aus dem Abschnitt.")
    user = (
        f"Erstelle genau {n} Frage-Antwort-Paare (Q&A) AUF DEUTSCH.\n"
        "Gib NUR diese Form aus, ohne zusätzliche Zeichen oder Erklärungen:\n"
        "Q: ...?\nA: ...\nQ: ...?\nA: ...\n...\n\n"
        "Beispiel:\n" + FEWSHOT + "\n"
        "Abschnitt:\n" + section + "\n"
    )
    return (system, user)

# --- Generate and extract with ONE regex (no JSON at all) ---
OUT_QA = Path("domain_qa.jsonl")
all_qas = []

for i, seg in enumerate(segments, 1):
    sys, usr = make_prompts(seg, n=3)
    out = chat(sys, usr, temperature=0.4)
    # extract: Q: <q>  A: <a>   (greedy until next Q: or end)
    pairs = re.findall(r"Q:\s*(.+?)\s*A:\s*(.+?)(?=\nQ:|\Z)", out, flags=re.S)
    # cleanup + store
    for q, a in pairs:
        q = re.sub(r"\s+", " ", q).strip()
        a = re.sub(r"\s+", " ", a).strip()
        if q.endswith("?") and len(q) > 10 and len(a) > 10:
            all_qas.append({"question": q, "answer": a})
    print(f"Segment {i}: found {len(pairs)} pairs")
    print("Pairs:",pairs)

with OUT_QA.open("w", encoding="utf-8") as f:
    for qa in all_qas:
        f.write(json.dumps(qa, ensure_ascii=False) + "\n")

print(f"Wrote {len(all_qas)} Q&As → {OUT_QA}")

Using 62 segments.
Segment 1: found 7 pairs
Pairs: [('...?', '...'), ('...?', '...\n...\n\nBeispiel:'), ('Was bedeutet Versicherungspflicht?', 'Sie ist die gesetzliche Verpflichtung, krankenversichert zu sein.'), ('Wer ist pflichtversichert?', 'Arbeitnehmer, Studierende und Rentner, sofern gesetzliche Kriterien erfüllt sind.\n\nAbschnitt:\nRatgeber Krankenversicherung www.bundesgesundheitsministerium.de Diese Druckschrift wird im Rahmen der Öffentlichkeitsarbeit des Bundesministeriums für Gesundheit kostenlos herausgegeben. Sie darf weder von Parteien noch von Wahlwerbern oder Wahlhelfern während des Wahlkampfes zum Zwecke der Wahlwerbung verwendet werden. Dies gilt für Europa-, Bundestags-, Landtags- und Kommunalwahlen. Ratgeber  Krankenversicherung Alles, was Sie zum Thema Krankenversicherung  wissen\xa0müssen\n\n\n Ratgeber  Krankenversicherung Alles, was Sie zum Thema Krankenversicherung wissen müssen\n viele Menschen machen die Erfahrung, wie sich das Leben plötzlich verändern kan